# 02: Multi-Phase Binding Kinetics Modelling

## Overview
This notebook simulates the molecular capture efficiency at the LFA test line using the Langmuir binding model. It models the competition between two Nanobodies (Nbs) and two Monoclonal Antibodies (mAbs) for HIV p24 capsid protein target across three distinct operational phases of the automated assay.

## Device Context
The automated LFIA separates the detection process into three sequential phases, timed by the fluid dynamics from Notebook 01:

| Phase | Time window | Event |
|-------|-------------|-------|
| **Phase 1** | t = 0 – 120 s | Target binds to PtNC–Ab conjugate in sample channel |
| **Phase 2** | t = 120 – 300 s | Target–conjugate complex binds to printed Ab at test line |
| **Phase 3** | t = 300 – 600 s | PBS wash arrives; unbound complex is removed by dissociation |

Signal amplification (DAB/PtNC catalysis) occurs after Phase 3 — so the fraction of complex **retained after the wash** directly determines the final signal.

## Binding Model
Under continuous lateral flow, the analyte concentration at the binding surface is maintained approximately constant, allowing the complex formation to be treated as **pseudo-first-order kinetics** (Landry et al., 2015).

**Association phase** (Phases 1 & 2):
$$\theta = \frac{[A]}{K_D + [A]} \left[1 - e^{-(k_a[A] + k_d)t}\right] \quad \cdots (1)$$

where $\theta = R_t / R_{max}$ is the fraction of binding, $[A]$ is the analyte concentration, $K_D = k_d/k_a$ is the equilibrium dissociation constant, and $k_a$, $k_d$ are the association and dissociation rate constants.

**Dissociation phase** (Phase 3): When $[A] \rightarrow 0$ upon PBS wash arrival, the bound complex decays by first-order kinetics:
$$\theta(t) = \theta_0 \cdot e^{-k_d t} \quad \cdots (2)$$

## Kinetic Parameters
All $k_a$, $k_d$, and $K_D$ values are taken from **Gray et al., 2017** (Table S3), which reports SPR measurements of Nb/mAb binding to HIV-1 p24 capsid protein:

> Gray, E. R. et al. *Unravelling the Molecular Basis of High Affinity Nanobodies against HIV p24.* ACS Infect. Dis. **2017**, 3, 479–491. DOI: [10.1021/acsinfecdis.6b00189](https://doi.org/10.1021/acsinfecdis.6b00189)

**Note:** Values are reported as medians with 95% confidence intervals from SPR experiments. $K_D$ is measured at equilibrium and may differ slightly from $k_d/k_a$ (which is typical for independently fitted SPR parameters).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## Configuration
All kinetic parameters and assay timing defined in one place.

In [ ]:
# =============================================================================
# KINETIC PARAMETERS — Gray et al. (2017), Table S3.
# =============================================================================
BINDERS = {
    "Nb 59H10":    {"KD": 6.9e-10, "ka": 3.7e5, "kd": 2.9e-4},  # Nanobody
    "Nb 37E7":     {"KD": 1.1e-7,  "ka": 4.2e4, "kd": 6.4e-3},  # Nanobody
    "mAb NIH-3537":{"KD": 2.2e-8,  "ka": 2.1e3, "kd": 1.7e-4},  # Monoclonal Ab
    "mAb CPHIV-1/2":{"KD": 1.6e-9, "ka": 4.5e4, "kd": 2.0e-4}, # Monoclonal Ab
}

# =============================================================================
# ASSAY PARAMETERS
# Phase 1 exploration.
# =============================================================================
CONC_NM   = 12.5         # Initial target (p24) concentration (nM)
CONC_M    = CONC_NM * 1e-9  # In Molarity

T_PHASE1  = 120          # Phase 1 duration (s): sample channel → test line
T_PHASE2  = 180          # Phase 2 duration (s): binding at test line (120–300 s)
T_PHASE3  = 300          # Phase 3 duration (s): PBS wash (300–600 s)

print("Configuration loaded.")
print(f"  Target concentration: {CONC_NM} nM")
print(f"  Phase 1: {T_PHASE1}s | Phase 2: {T_PHASE2}s | Phase 3: {T_PHASE3}s")
print()
print(f"{'Binder':<18} {'KD (M)':>12}  {'ka (M-1s-1)':>14}  {'kd (s-1)':>10}")
print("-" * 62)
for name, p in BINDERS.items():
    print(f"{name:<18} {p['KD']:>12.2e}  {p['ka']:>14.2e}  {p['kd']:>10.2e}")

## Helper Functions

In [ ]:
def langmuir_assoc(KD, ka, kd, t_array, conc_M):
    """Fraction of binding during association phase — Equation (1).

    Parameters
    ----------
    KD      : float   — equilibrium dissociation constant (M)
    ka      : float   — association rate constant (M⁻¹s⁻¹)
    kd      : float   — dissociation rate constant (s⁻¹)
    t_array : ndarray — time points (s)
    conc_M  : float   — analyte concentration (M)

    Returns
    -------
    theta : ndarray — fraction of binding (0–1)
    """
    theta_eq = conc_M / (KD + conc_M)
    k_obs    = ka * conc_M + kd
    return theta_eq * (1 - np.exp(-k_obs * t_array))


def langmuir_dissoc(theta0, kd, t_array):
    """Fraction of binding remaining during dissociation phase — Equation (2).

    Parameters
    ----------
    theta0  : float   — fraction of binding at the start of dissociation
    kd      : float   — dissociation rate constant (s⁻¹)
    t_array : ndarray — time points relative to wash start (s)

    Returns
    -------
    theta : ndarray — fraction of binding remaining (0–1)
    """
    return theta0 * np.exp(-kd * t_array)


print("Helper functions defined.")

---
## Section 1 — Model Exploration: Concentration Dependence (Phase 1)
Before applying the assay timeline, we examine how target concentration affects binding speed and equilibrium level for each binder. This reproduces the exploration in the original legacy notebooks and establishes 12.5 nM as the working concentration.

In [ ]:
concs_nM  = [1.6, 3.1, 6.3, 12.5, 25, 50, 100]
times_exp = np.arange(0, 610, 10, dtype=float)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Phase 1 — Concentration Dependence of Langmuir Binding (0–600 s)", fontsize=12)

for ax, (name, p) in zip(axes.flat, BINDERS.items()):
    for c_nM in concs_nM:
        c_M   = c_nM * 1e-9
        theta = langmuir_assoc(p['KD'], p['ka'], p['kd'], times_exp, c_M)
        ax.plot(times_exp, theta, label=f"{c_nM} nM")

    ax.axvline(x=120, color='red', linestyle='--', alpha=0.7, label='Phase 1 end (120 s)')
    ax.set_title(name)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Fraction of binding (θ)")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 2 — Phase 1: Target Binding to PtNC–Ab Conjugate (t = 0 to 120 s)
At the diagnostic concentration of **12.5 nM**, we compare the fraction of binding achieved by all four binders at the end of Phase 1 (t = 120 s), when the target–conjugate complex reaches the test line.

In [ ]:
times_p1 = np.arange(0, T_PHASE1 + 10, 10, dtype=float)

fig, ax = plt.subplots(figsize=(9, 5))
theta1_results = {}

for name, p in BINDERS.items():
    theta = langmuir_assoc(p['KD'], p['ka'], p['kd'], times_p1, CONC_M)
    theta_at_120 = float(np.interp(120, times_p1, theta))
    theta1_results[name] = theta_at_120

    ax.plot(times_p1, theta, label=f"{name}")
    ax.scatter(120, theta_at_120, zorder=5)
    print(f"  {name:<20}: θ at 120 s = {theta_at_120:.4f}  ({theta_at_120*100:.2f}%)")

ax.axvline(x=120, color='red', linestyle='--', alpha=0.8, label='Phase 1 end (t = 120 s)')
ax.set_title(f"Phase 1 — Fraction of Binding vs Time ([A] = {CONC_NM} nM)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Fraction of binding (θ)")
ax.set_xlim(0, 130)
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 3 — Phase 2: Complex Binding at the Test Line (t = 120 to 300 s)
The initial concentration for Phase 2 is the fraction of complex formed in Phase 1 multiplied by the original target concentration: $[A]_2 = [A]_1 \times \theta_1$. This complex then binds to the printed antibody/nanobody-biotin at the test line over the next 180 s.

In [ ]:
# Using CONC_NM explicitly here to make the dependency clear.

times_p2 = np.arange(0, T_PHASE2 + 10, 10, dtype=float)

fig, ax = plt.subplots(figsize=(9, 5))
theta2_results = {}
conc2_results  = {}

for name, p in BINDERS.items():
    conc2_nM = CONC_NM * theta1_results[name]   # effective complex concentration entering Phase 2
    conc2_M  = conc2_nM * 1e-9
    conc2_results[name] = conc2_nM

    theta = langmuir_assoc(p['KD'], p['ka'], p['kd'], times_p2, conc2_M)
    theta_at_180 = float(np.interp(180, times_p2, theta))
    theta2_results[name] = theta_at_180

    ax.plot(times_p2, theta, label=f"{name}")
    ax.scatter(180, theta_at_180, zorder=5)
    print(f"  {name:<20}: [A]₂ = {conc2_nM:.4f} nM | θ at 180 s = {theta_at_180:.4f}  ({theta_at_180*100:.2f}%)")

ax.axvline(x=180, color='red', linestyle='--', alpha=0.8, label='Phase 2 end (Δt = 180 s)')
ax.set_title("Phase 2 — Complex Binding at Test Line (relative time, 0–180 s)")
ax.set_xlabel("Time elapsed in Phase 2 (s)")
ax.set_ylabel("Fraction of binding (θ)")
ax.set_ylim(0)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 4 — Phase 3: PBS Wash / Dissociation (t = 300 to 600 s)
When the PBS buffer arrives at the test line, analyte concentration drops to zero and the bound complex dissociates by first-order kinetics (Equation 2). The fraction of complex **retained after 300 s of wash** is the quantity available for signal amplification by the DAB/PtNC system.

In [ ]:
times_p3 = np.arange(0, T_PHASE3 + 10, 10, dtype=float)

fig, ax = plt.subplots(figsize=(9, 5))
theta3_results = {}

for name, p in BINDERS.items():
    theta_wash = langmuir_dissoc(theta2_results[name], p['kd'], times_p3)
    theta_at_300 = float(np.interp(300, times_p3, theta_wash))
    theta3_results[name] = theta_at_300
    retention = (theta_at_300 / theta2_results[name] * 100) if theta2_results[name] > 0 else 0

    ax.plot(times_p3, theta_wash, label=f"{name}")
    ax.scatter(300, theta_at_300, zorder=5)
    print(f"  {name:<20}: θ after 300 s wash = {theta_at_300:.4f}  (retained {retention:.1f}%)")

ax.axvline(x=300, color='red', linestyle='--', alpha=0.8, label='Phase 3 end (Δt = 300 s)')
ax.set_title("Phase 3 — Dissociation During PBS Wash (relative time, 0–300 s)")
ax.set_xlabel("Time elapsed in Phase 3 (s)")
ax.set_ylabel("Fraction of binding remaining (θ)")
ax.set_ylim(0)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 5 — Full 3-Phase Timeline Summary
Plotting all three phases on a single absolute time axis (0–600 s) to visualise the complete assay workflow.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

colours = plt.rcParams['axes.prop_cycle'].by_key()['color']

for i, (name, p) in enumerate(BINDERS.items()):
    colour = colours[i]

    # Phase 1: t = 0–120 s
    t1 = np.arange(0, 121, 5, dtype=float)
    th1 = langmuir_assoc(p['KD'], p['ka'], p['kd'], t1, CONC_M)

    # Phase 2: t = 120–300 s
    t2_rel = np.arange(0, 181, 5, dtype=float)
    conc2  = conc2_results[name] * 1e-9
    th2    = langmuir_assoc(p['KD'], p['ka'], p['kd'], t2_rel, conc2)

    # Phase 3: t = 300–600 s
    t3_rel = np.arange(0, 301, 5, dtype=float)
    th3    = langmuir_dissoc(theta2_results[name], p['kd'], t3_rel)

    ax.plot(t1,          th1,  color=colour, label=name)
    ax.plot(t2_rel + 120, th2,  color=colour, linestyle='--')
    ax.plot(t3_rel + 300, th3,  color=colour, linestyle=':')

ax.axvline(x=120, color='gray',  linestyle='--', alpha=0.6, label='Phase 1→2 (t=120 s)')
ax.axvline(x=300, color='gray',  linestyle=':',  alpha=0.6, label='Phase 2→3 (t=300 s)')
ax.axvspan(0,   120, alpha=0.04, color='blue',   label='Phase 1 (association)')
ax.axvspan(120, 300, alpha=0.04, color='green',  label='Phase 2 (test line binding)')
ax.axvspan(300, 600, alpha=0.04, color='orange', label='Phase 3 (PBS wash)')
ax.set_title("Full 3-Phase Binding Kinetics — Automated LFIA Timeline")
ax.set_xlabel("Absolute time (s)")
ax.set_ylabel("Fraction of binding (θ)")
ax.set_xlim(0, 600)
ax.set_ylim(0)
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 6 — Conclusion: Ranking by Signal Retained After Wash

In [ ]:
print("Final comparison — fraction of complex retained at test line after PBS wash:")
print(f"  [p24] = {CONC_NM} nM | Source: Gray et al. (2017)")
print()
print(f"{'Rank':<6} {'Binder':<20} {'θ Phase 1':>10} {'θ Phase 2':>10} {'θ Phase 3':>10} {'% retained':>12}")
print("-" * 72)

ranked = sorted(BINDERS.keys(), key=lambda n: theta3_results[n], reverse=True)
for rank, name in enumerate(ranked, 1):
    th1  = theta1_results[name]
    th2  = theta2_results[name]
    th3  = theta3_results[name]
    ret  = (th3/th2*100) if th2 > 0 else 0
    tag  = " ← selected" if name == "Nb 59H10" else ""
    print(f"  {rank}    {name:<20} {th1:>10.4f} {th2:>10.4f} {th3:>10.4f} {ret:>11.1f}%{tag}")

print()
print("Conclusion:")
print("  Nb 59H10 achieves the highest absolute signal retention after the PBS wash,")
print("  owing to its high ka (fast on-rate) and low kd (slow off-rate), giving")
print(f"  KD = {BINDERS['Nb 59H10']['KD']:.1e} M — the tightest binder of the four candidates.")
print("  Its combination of Phase 1 binding efficiency and Phase 3 wash retention")
print("  makes it the optimal candidate for PtNC conjugation in the automated LFIA.")